In [1]:
# Balloon Loan Calculator
# This calculator can be used to compute either the regular payment for
# a balloon loan, or the present value of a balloon loan given the amount
# of the regular payments.
#
# How to use:
#   - To compute the periodic payment: enter the loan amount, leave the
#     periodic payment field at 0, and click 'Compute unknown variable'.
#   - To compute the present value: enter the periodic payment amount,
#     leave the loan amount at 0, and click 'Compute unknown variable'.
#   - In both cases, also fill in the interest rate per period, the total
#     number of payments, and the balloon-payment amount.
#
# Note: 'Interest rate per period' is entered as a decimal
# (e.g. 0.005 means 0.5% per period).

import tkinter as tk
import tkinter.ttk as ttk

root = tk.Tk()
root.title('Balloon Loan Calculator')

output_widgets = {}  # holds result Text/Label widgets so we can clear or replace them safely

def quit_app():
    root.destroy()

def remove_outputs():
    # Destroy any output widgets that have been previously created
    for key in ('label', 'text'):
        widget = output_widgets.get(key)
        if widget is not None and widget.winfo_exists():
            widget.destroy()
    output_widgets.clear()

def clear():
    # Reset all input fields and remove any previously displayed result
    remove_outputs()
    for entry in (q1, q2, q3, q5, q6):
        entry.delete(0, tk.END)
    for var in (w, x, y, z1, z2):
        var.set(0)

def show_result(label_text, value):
    # Replace any prior result with a fresh label + text widget
    remove_outputs()
    lbl = tk.Label(root, text=label_text)
    lbl.grid(row=9, column=1, padx=3, pady=5, sticky='w')
    txt = tk.Text(root, height=1, width=15)
    txt.insert(tk.END, '${:,.2f}'.format(value))
    txt.grid(row=9, column=2, padx=3, sticky='w')
    output_widgets['label'] = lbl
    output_widgets['text']  = txt

def show_message(msg):
    # Used when inputs are missing or contradictory
    remove_outputs()
    lbl = tk.Label(root, text=msg, fg='red')
    lbl.grid(row=9, column=1, columnspan=2, padx=3, pady=5, sticky='w')
    output_widgets['label'] = lbl

def read_inputs():
    # All DoubleVars start at 0; if the user blanks out a field, get() raises TclError
    def safe(var):
        try:
            return float(var.get())
        except (tk.TclError, ValueError):
            return 0.0
    p   = safe(w)    # current value (principal) of loan
    r   = safe(x)    # interest rate per payment period (decimal)
    n   = safe(y)    # total number of payments
    b   = safe(z1)   # amount of the balloon payment
    epa = safe(z2)   # amount of each recurring payment
    return p, r, n, b, epa

def compute_var():
    # Compute whichever of P or epa was left at 0.
    p, r, n, b, epa = read_inputs()

    if n <= 0:
        show_message("Please enter a total number of payments greater than 0.")
        return
    if p == 0 and epa == 0:
        show_message("Enter either the loan amount or the periodic payment (leave the other at 0).")
        return
    if p != 0 and epa != 0:
        show_message("Both loan amount and periodic payment are filled in. Set one to 0.")
        return

    g = (1 + r) ** n

    if epa == 0:
        # Solve for the periodic payment given the loan amount
        if r == 0:
            epa_out = (p - b) / n
        else:
            epa_out = (r * p * g - r * b) / (g - 1)
        show_result("The payment amount per period is:", epa_out)
    else:
        # Solve for the present value given the periodic payment
        if r == 0:
            p_out = epa * n + b
        else:
            p_out = (epa * (g - 1) + r * b) / (r * g)
        show_result("The current value of the loan is:", p_out)

# Input
tk.Label(root, text="Amount of loan\n(leave at 0 if computing present value):").\
    grid(row=1, column=1, padx=3, sticky='w')
w  = tk.DoubleVar()
q1 = tk.Entry(root, textvariable=w)
q1.grid(row=1, column=2, padx=5, pady=3, sticky='w')

tk.Label(root, text="Interest rate per period (decimal, e.g. 0.005):").\
    grid(row=2, column=1, padx=3, sticky='w')
x  = tk.DoubleVar()
q2 = tk.Entry(root, textvariable=x)
q2.grid(row=2, column=2, padx=5, pady=3, sticky='w')

tk.Label(root, text="Total number of payments:").grid(row=3, column=1, padx=3, sticky='w')
y  = tk.DoubleVar()
q3 = tk.Entry(root, textvariable=y)
q3.grid(row=3, column=2, padx=5, pady=3, sticky='w')

tk.Label(root, text="Amount of the balloon payment:").grid(row=5, column=1, padx=3, sticky='w')
z1 = tk.DoubleVar()
q5 = tk.Entry(root, textvariable=z1)
q5.grid(row=5, column=2, padx=5, pady=3, sticky='w')

tk.Label(root, text="Amount of each periodic payment\n(leave at 0 if computing the periodic payment):").\
    grid(row=6, column=1, padx=3, sticky='w')
z2 = tk.DoubleVar()
q6 = tk.Entry(root, textvariable=z2)
q6.grid(row=6, column=2, padx=5, pady=3, sticky='w')

# Buttons
tk.Button(root, text="Compute unknown variable", command=compute_var).\
    grid(row=8, column=1, padx=3, pady=7)

tk.Button(root, text='Clear', command=clear).grid(row=23, column=1, pady=5)
tk.Button(root, text='Quit',  command=quit_app).grid(row=23, column=2, padx=3, pady=5, sticky='w')

# Bring the window to the front when it opens, but don't keep it always-on-top.
# (Same three-line pattern used in amortizations.ipynb.)
root.lift()
root.attributes('-topmost', True)
root.after_idle(root.attributes, '-topmost', False)

root.mainloop()